# Ch01-09: 대용량 데이터 처리 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

메모리 최적화 전후 비교

In [ ]:
import numpy as np, pandas as pd, time

np.random.seed(42); n=2_000_000
df = pd.DataFrame({
    'id': np.arange(n), 'cat': np.random.choice([f'cat_{i}' for i in range(50)], n),
    'v1': np.random.normal(100,20,n), 'v2': np.random.randint(0,100,n), 'flag': np.random.randint(0,2,n),
})

before = df.memory_usage(deep=True).sum()
t0 = time.time(); _ = df.groupby('cat')['v1'].mean(); t_before = time.time()-t0

for c in df.select_dtypes('int64').columns:
    mn,mx = df[c].min(), df[c].max()
    if mn>=0 and mx<255: df[c]=df[c].astype('uint8')
    elif mn>-32768 and mx<32767: df[c]=df[c].astype('int16')
    else: df[c]=df[c].astype('int32')
for c in df.select_dtypes('float64').columns: df[c]=df[c].astype('float32')
for c in df.select_dtypes('object').columns: df[c]=df[c].astype('category')

after = df.memory_usage(deep=True).sum()
t0 = time.time(); _ = df.groupby('cat')['v1'].mean(); t_after = time.time()-t0

print(f"메모리: {before/1e6:.1f}MB → {after/1e6:.1f}MB ({100*(1-after/before):.1f}% 절감)")
print(f"GroupBy 시간: {t_before*1000:.1f}ms → {t_after*1000:.1f}ms")


**결과 해석**: 다운캐스팅과 범주형 변환으로 60-80% 메모리 절감. 캐시 효율 향상으로 연산 속도도 개선된다.

---
## 문제 2 풀이

Welford 온라인 통계량

In [ ]:
import numpy as np

class OnlineStats:
    def __init__(self):
        self.n = 0; self.mean = 0; self.M2 = 0; self.min_v = np.inf; self.max_v = -np.inf
    def update(self, x):
        if np.isscalar(x): x = [x]
        for val in x:
            self.n += 1; delta = val-self.mean
            self.mean += delta/self.n; delta2 = val-self.mean
            self.M2 += delta*delta2
            self.min_v = min(self.min_v, val); self.max_v = max(self.max_v, val)
    @property
    def variance(self): return self.M2/(self.n-1) if self.n>1 else 0
    @property
    def std(self): return np.sqrt(self.variance)
    def summary(self): return f"n={self.n}, mean={self.mean:.4f}, std={self.std:.4f}, min={self.min_v:.4f}, max={self.max_v:.4f}"

# 검증: 청크 처리 vs 전체
np.random.seed(42); data = np.random.normal(42, 7, 100000)
os = OnlineStats()
chunk_size = 1000
for i in range(0, len(data), chunk_size):
    os.update(data[i:i+chunk_size])

print(f"Online:  {os.summary()}")
print(f"NumPy:   n={len(data)}, mean={data.mean():.4f}, std={data.std(ddof=1):.4f}, min={data.min():.4f}, max={data.max():.4f}")
print(f"일치: mean차이={abs(os.mean-data.mean()):.2e}, std차이={abs(os.std-data.std(ddof=1)):.2e}")


**결과 해석**: Welford 알고리즘으로 단일 패스 + O(1) 메모리로 정확한 평균/분산 계산. 수치 안정성이 단순 합산 방식보다 우수.

---
## 문제 3 풀이

Pandas 최적화 벤치마크

In [ ]:
import numpy as np, pandas as pd, time

np.random.seed(42); n=1_000_000
df = pd.DataFrame({
    'key1': np.random.choice([f'k{i}' for i in range(100)], n),
    'key2': np.random.choice([f'g{i}' for i in range(20)], n),
    'val': np.random.randn(n).astype('float32'),
    'id': np.arange(n, dtype='int32'),
})
df2 = pd.DataFrame({'key1': [f'k{i}' for i in range(100)], 'info': np.random.randn(100)})

ops = {}

t0=time.time(); _ = df.groupby('key1')['val'].agg(['mean','std','count']); ops['groupby']=time.time()-t0
t0=time.time(); _ = df.merge(df2, on='key1'); ops['join']=time.time()-t0
t0=time.time(); _ = df[df['val']>0]; ops['filter']=time.time()-t0
t0=time.time(); _ = df.sort_values('val'); ops['sort']=time.time()-t0

print(f"{'Operation':12s} {'Time(ms)':>10s}")
for op, t in ops.items(): print(f"{op:12s} {t*1000:10.1f}")

# 카테고리 최적화 효과
df_opt = df.copy(); df_opt['key1']=df_opt['key1'].astype('category')
t0=time.time(); _ = df_opt.groupby('key1')['val'].mean(); t_cat = time.time()-t0
t0=time.time(); _ = df.groupby('key1')['val'].mean(); t_str = time.time()-t0
print(f"\nGroupBy: string={t_str*1000:.1f}ms, category={t_cat*1000:.1f}ms")


**결과 해석**: 범주형 변환으로 GroupBy 속도 향상. Polars는 병렬 처리+쿼리 최적화로 추가 성능 이점.

---
## 문제 4 풀이

스트리밍 GroupBy 구현

In [ ]:
import numpy as np, pandas as pd

class StreamingGroupBy:
    def __init__(self):
        self.groups = {}  # {key: {'n':0, 'mean':0, 'M2':0}}
    def update(self, chunk_df, key_col, val_col):
        for key, grp in chunk_df.groupby(key_col):
            vals = grp[val_col].values
            if key not in self.groups:
                self.groups[key] = {'n':0, 'mean':0.0, 'M2':0.0}
            g = self.groups[key]
            for v in vals:
                g['n'] += 1; d = v-g['mean']
                g['mean'] += d/g['n']; d2 = v-g['mean']
                g['M2'] += d*d2
    def result(self):
        rows = []
        for key, g in self.groups.items():
            var = g['M2']/(g['n']-1) if g['n']>1 else 0
            rows.append({'key':key, 'count':g['n'], 'mean':g['mean'], 'std':np.sqrt(var)})
        return pd.DataFrame(rows).sort_values('key').reset_index(drop=True)

# 검증
np.random.seed(42); n=500000
df = pd.DataFrame({'key':np.random.choice(['A','B','C','D','E'], n), 'val':np.random.randn(n)})

# 스트리밍
sgb = StreamingGroupBy()
chunk_size = 10000
for i in range(0, n, chunk_size):
    sgb.update(df.iloc[i:i+chunk_size], 'key', 'val')
res_stream = sgb.result()

# 전체
res_full = df.groupby('key')['val'].agg(['count','mean','std']).reset_index()
res_full.columns = ['key','count','mean','std']

print("Streaming vs Full:")
merged = res_stream.merge(res_full, on='key', suffixes=('_stream','_full'))
for _, r in merged.iterrows():
    print(f"  {r['key']}: mean diff={abs(r['mean_stream']-r['mean_full']):.2e}, std diff={abs(r['std_stream']-r['std_full']):.2e}")


**결과 해석**: 스트리밍 GroupBy가 전체 로드와 동일한 결과를 O(그룹 수) 메모리로 달성. Welford 기반으로 수치 안정성 보장.

---
## 확장 토론

### 대용량 처리 전략

1. 메모리 최적화 우선 (다운캐스팅, 범주형)
2. 스트리밍/청크 처리 (Welford 기반)
3. Polars/Dask 활용 (병렬+지연평가)
4. 데이터베이스 연계 (SQL pushdown)